# 🚀 LumiLearn 一键部署教程

> 从远程服务器服务器零基础部署到内网多设备使用

---

## 你将学到什么

| 步骤 | 目标 | 耗时 |
|------|------|------|
| Step 1 | 克隆 GitHub 代码仓库 | 1 分钟 |
| Step 2 | 一键部署脚本运行 | 2-5 分钟 |
| Step 3 | Ollama 模型配置 | 10-20 分钟（下载） |
| Step 4 | 验证所有 API 端点 | 1 分钟 |
| Step 5 | 浏览器访问 LumiTerminal | 即开即用 |

## 架构速览

```
远程服务器服务器 (192.168.2.xx)
├── Ollama :11434        ← 本地大模型推理
├── LumiLearn :18080     ← LumiTerminal 终端 + API
├── :18081               ← 纯 REST API
└── :18082               ← 模型管理
         ↓
    局域网 (192.168.xx.0/24)
         ↓
  笔记本 / 平板 / 手机 / 其他PC
```

> 💡 **核心概念**：LumiLearn 采用五层架构（L1-L5），从自研 Transformer 到多模态交互，三端口部署让终端页面、API 服务、模型管理各司其职。内网任何设备打开浏览器即可访问。

## 环境准备

先确认你的服务器环境是否满足要求：

- **Python**: 3.9+
- **Git**: 用于克隆代码
- **磁盘空间**: 至少 20GB 可用（模型文件较大）
- **内存**: 8GB+ 用于 7B 模型推理

运行下面的代码检查环境：

In [ ]:
import sys, shutil, subprocess, os

print("=" * 50)
print("🔍 环境检查")
print("=" * 50)

print(f"Python 版本: {sys.version}")
assert sys.version_info >= (3, 9), "❌ Python 版本过低，需要 3.9+"
print("✅ Python 版本满足要求 (>= 3.9)")

total, used, free = shutil.disk_usage("/")
free_gb = free // (2**30)
print(f"磁盘空间: 空闲 {free_gb} GB / 总计 {total // (2**30)} GB")
if free_gb < 20:
    print("⚠️ 磁盘空间不足 20GB，模型下载可能失败")
else:
    print("✅ 磁盘空间充足")

result = subprocess.run(["git", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"Git: {result.stdout.strip()}")
    print("✅ Git 已安装")
else:
    print("❌ Git 未安装，请先安装: apt install git")

print(f"当前目录: {os.getcwd()}")
print(f"操作系统: {sys.platform}")
print("=" * 50)
print("环境检查完毕！")

## Step 1: 克隆代码仓库

从 GitHub 克隆 LumiLearn 完整代码。项目包含：
- `framework/` — 五层核心框架（L1-L5）
- `models/distil/` — 三大蒸馏模型（Teacher/Critic/Evolver）
- `config/` — 部署与模型配置
- `remote/` — 远程服务器服务器专用部署文件

> 📖 **背景**：LumiLearn 是端到端的 AI 教育平台，融合了自研微型 Transformer、知识蒸馏管线、多模态交互终端和费曼教学引擎。

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/lumilearn/lumilearn.git"

if os.path.exists("lumilearn"):
    print("⚠️ lumilearn 目录已存在，跳过克隆")
    os.chdir("lumilearn")
else:
    print(f"📦 正在克隆 {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"❌ 克隆失败: {result.stderr}")
        print("💡 请检查网络连接或 GitHub 仓库是否公开")
    else:
        print("✅ 克隆成功！")
        os.chdir("lumilearn")

print(f"📂 当前工作目录: {os.getcwd()}")

result = subprocess.run(
    ["ls", "-la"] if sys.platform != "win32" else ["cmd", "/c", "dir"],
    capture_output=True, text=True
)
print(result.stdout[:2000])

## Step 2: 一键部署

运行一键部署脚本。这会：
1. 安装 Python 依赖（`requirements.txt`）
2. 初始化配置文件
3. 启动三端口服务器（18080/18081/18082）

`--skip-model` 参数跳过模型下载（后续手动配置 Ollama）。

> ⚠️ **注意**：首次运行可能需要 2-5 分钟安装依赖。如果模型下载已包含在脚本中，完整部署可能需要 10-20 分钟。

In [ ]:
import subprocess, os

script = "deploy.sh" if sys.platform != "win32" else "deploy.bat"

if os.path.exists(script):
    print(f"🚀 运行一键部署脚本: {script} --skip-model")
    print("=" * 50)
    if sys.platform != "win32":
        result = subprocess.run(
            ["bash", script, "--skip-model"],
            capture_output=True, text=True,
            timeout=600
        )
    else:
        result = subprocess.run(
            [script, "--skip-model"],
            capture_output=True, text=True,
            timeout=600, shell=True
        )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print(f"⚠️ 部署脚本返回非零退出码: {result.returncode}")
        print("请检查错误信息并手动排查")
else:
    print(f"ℹ️ {script} 不存在，手动安装依赖...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--user"],
        capture_output=True, text=True,
        timeout=600
    )
    if result.returncode == 0:
        print("✅ 依赖安装成功")
    else:
        print(f"❌ 依赖安装失败: {result.stderr[:500]}")

print("=" * 50)
print("📋 部署脚本执行完毕")

## Step 3: Ollama 模型配置

LumiLearn 依赖 Ollama 进行本地模型推理。核心模型：
- **lumilearn-7b**: 蒸馏后的教学模型（基于 qwen2.5:7b 微调）
- **qwen2.5:7b**: 基础中文模型（兜底使用）

先检查 Ollama 是否运行，然后拉取模型。

> 📖 **双模型架构**：复杂推理（费曼深度讲解、数学证明）走大模型 API，日常问答走本地 7B 模型。模型路由器（`framework/core/router.py`）自动分流。

In [ ]:
import subprocess, json as j

print("🔍 检查 Ollama 服务状态...")

try:
    result = subprocess.run(
        ["curl", "-s", "http://localhost:11434/api/tags"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0 and result.stdout.strip():
        data = j.loads(result.stdout)
        models = data.get("models", [])
        if models:
            print(f"✅ Ollama 运行中，已安装 {len(models)} 个模型:")
            for m in models[:10]:
                print(f"   📦 {m.get('name', 'unknown')} ({m.get('size', '?')})")
        else:
            print("✅ Ollama 运行中，但尚无模型")
    else:
        print("⚠️ Ollama 似乎未运行或响应异常")
except (subprocess.TimeoutExpired, j.JSONDecodeError):
    print("❌ Ollama 未运行！请先启动: ollama serve")

print("\n📥 拉取基础模型 qwen2.5:7b（如已存在则跳过）...")
print("   ⏳ 首次下载约需 10-20 分钟（~4.5GB）")

result = subprocess.run(
    ["ollama", "pull", "qwen2.5:7b"],
    capture_output=True, text=True, timeout=1800
)
if result.returncode == 0:
    print("✅ 模型拉取成功")
else:
    print(f"⚠️ 模型拉取可能失败: {result.stderr[-300:]}")
    print("💡 可以稍后手动执行: ollama pull qwen2.5:7b")

print("\n📥 拉取蒸馏教学模型 lumilearn-7b ...")
result = subprocess.run(
    ["ollama", "pull", "lumilearn-7b"],
    capture_output=True, text=True, timeout=1800
)
print("✅ 模型配置完毕" if result.returncode == 0 else "⚠️ 请后续手动拉取: ollama pull lumilearn-7b")

## Step 4: 验证服务

LumiLearn 暴露了丰富的 API 端点。我们来逐项验证：

| 端点 | 用途 |
|------|------|
| `/health` | 健康检查 |
| `/api/status` | 服务器状态 |
| `/api/chat` | 对话接口 |
| `/api/models` | 模型列表 |
| `/api/feynman/explain` | 费曼教学（五步讲解） |
| `/api/review` | 讲解质量审查（4 维度评分） |
| `/api/resources` | 资源搜索（RAG 摘要） |

> 📖 **费曼教学引擎**采用五步教学法：现象引入 → 认知冲突 → 思维模型 → 自主推导 → 30秒测试。

In [ ]:
import requests, json as j, sys

base = "http://192.168.2.xx:18080"

print("=" * 60)
print("🔬 API 端点全面验证")
print("=" * 60)

tests = [
    ("健康检查", "GET", "/health", None),
    ("服务状态", "GET", "/api/status", None),
    ("对话接口", "POST", "/api/chat",
     {"messages": [{"role": "user", "content": "你好"}]}),
    ("模型列表", "GET", "/api/models", None),
    ("费曼讲解", "POST", "/api/feynman/explain",
     {"topic": "勾股定理"}),
    ("质量审查", "POST", "/api/review",
     {"content": "勾股定理是a²+b²=c²"}),
    ("资源搜索", "POST", "/api/resources",
     {"query": "二次函数", "category": "math"}),
]

passed = 0
failed = 0

for name, method, path, body in tests:
    try:
        url = base + path
        if method == "GET":
            r = requests.get(url, timeout=10)
        else:
            r = requests.post(url, json=body, timeout=30)
        icon = "✅" if r.status_code == 200 else "❌"
        if r.status_code == 200:
            passed += 1
            detail = ""
            try:
                resp = r.json()
                detail = str(resp)[:80]
            except:
                detail = r.text[:80]
            print(f"{icon} {name:8s} | {method:4s} {path:25s} → {r.status_code} | {detail}")
        else:
            failed += 1
            print(f"{icon} {name:8s} | {method:4s} {path:25s} → {r.status_code}")
    except requests.exceptions.ConnectionError:
        failed += 1
        print(f"❌ {name:8s} | {method:4s} {path:25s} → 连接失败（服务未启动？）")
    except Exception as e:
        failed += 1
        print(f"❌ {name:8s} | {method:4s} {path:25s} → {e}")

print("=" * 60)
print(f"📊 结果: {passed} 通过 / {failed} 失败 / {len(tests)} 总计")
if failed == 0:
    print("🎉 所有端点验证通过！")
else:
    print("💡 请检查服务是否已启动: python -m framework.api.server --multi-port")

## Step 5: 浏览器访问 LumiTerminal

LumiTerminal 是多模态 AI 交互终端，支持：
- 🎤 **语音输入** (Ctrl+G 录音 → Whisper 转文字)
- 📷 **图片 OCR** (Ctrl+O 上传 → PaddleOCR 识别)
- 📋 **质量审查** (Ctrl+R 审查 AI 讲解)
- 🔍 **资源搜索** (Ctrl+F 搜索学习资源)
- 💬 **AI 对话**（支持流式 NDJSON）

内网任何设备的浏览器打开 `http://192.168.2.xx:18080` 即可使用。

In [ ]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="background:#1a1a2e;color:#0f0;padding:24px;border-radius:16px;font-family:Consolas,monospace;border:2px solid #00ff88">
  <h2 style="color:#00ff88;margin-top:0">🌐 LumiTerminal 已就绪</h2>
  <p>在浏览器打开：</p>
  <a href="http://192.168.2.xx:18080" target="_blank"
     style="color:#00bfff;font-size:28px;font-weight:bold;text-decoration:none;">
    http://192.168.2.xx:18080
  </a>
  <hr style="border-color:#333;margin:16px 0">
  <table style="color:#ccc;width:100%;font-size:14px">
    <tr><td style="padding:4px 8px">📱 笔记本</td><td>✅ 全功能交互</td></tr>
    <tr><td style="padding:4px 8px">📱 平板</td><td>✅ 触屏优化</td></tr>
    <tr><td style="padding:4px 8px">📱 手机</td><td>✅ 响应式布局</td></tr>
    <tr><td style="padding:4px 8px">🖥️ 其他PC</td><td>✅ 局域网访问</td></tr>
  </table>
  <p style="margin-top:16px;color:#888;font-size:12px">
    快捷键: Ctrl+G 录音 | Ctrl+O OCR | Ctrl+R 审查 | Ctrl+F 搜索 | Ctrl+K 清屏
  </p>
</div>
"""))

from IPython.display import IFrame
display(IFrame("http://192.168.2.xx:18080", width="100%", height="500"))

## 扩展：添加新模型

LumiLearn 的模型系统采用**适配器模式**，通过 `config/providers.yaml` 配置新模型，无需修改任何框架代码。

架构中的模型层（L2）支持三种接入方式：
1. **Ollama** — 本地模型（如 lumilearn-7b）
2. **OpenAI 兼容** — 任意兼容 OpenAI API 的模型（DeepSeek、通义千问）
3. **Custom Adapter** — 自定义适配器（星火、Claude、Gemini）

新模型配置后可通过 `/api/models/switch` 随时切换。

In [ ]:
config = """
# ============================================================
# config/providers.yaml — 模型提供者配置模板
# 填入 API Key 后复制到 config/providers.yaml 即可生效
# ============================================================

providers:
  # ──── OpenAI 兼容协议 ────
  - name: "deepseek"
    type: "openai_compatible"
    base_url: "https://api.deepseek.com/v1"
    api_key: "${DEEPSEEK_API_KEY}"
    models:
      - "deepseek-chat"
      - "deepseek-reasoner"

  - name: "qwen"
    type: "openai_compatible"
    base_url: "https://dashscope.aliyuncs.com/compatible-mode/v1"
    api_key: "${QWEN_API_KEY}"
    models:
      - "qwen-max"
      - "qwen-plus"

  # ──── 自定义适配器 ────
  - name: "spark"
    type: "custom"
    adapter: "spark_adapter"
    app_id: "${SPARK_APP_ID}"
    api_key: "${SPARK_API_KEY}"
    api_secret: "${SPARK_API_SECRET}"
    models:
      - "spark-4.0-ultra"

  - name: "claude"
    type: "custom"
    adapter: "claude_adapter"
    api_key: "${ANTHROPIC_API_KEY}"
    models:
      - "claude-opus-4"
      - "claude-sonnet-4"

  - name: "gemini"
    type: "custom"
    adapter: "gemini_adapter"
    api_key: "${GOOGLE_API_KEY}"
    models:
      - "gemini-ultra"
      - "gemini-pro"

  # ──── Ollama 本地 ────
  - name: "ollama"
    type: "ollama"
    base_url: "http://localhost:11434"
    models:
      - "lumilearn-7b"
      - "qwen2.5:7b"
"""

from IPython.display import display, HTML
display(HTML(f"""
<div style="background:#0d1117;color:#c9d1d9;padding:20px;border-radius:12px;font-family:Consolas,monospace;font-size:13px">
  <h3 style="color:#58a6ff">📋 config/providers.yaml 配置模板</h3>
  <p style="color:#8b949e">填入 API 密钥后复制到 config/providers.yaml 即可使用</p>
  <pre style="background:#161b22;padding:16px;border-radius:8px;overflow-x:auto;color:#c9d1d9">{config}</pre>
  <p style="color:#3fb950;margin-top:12px">
    ✅ 无需修改任何框架代码<br>
    ✅ 新模型通过 /api/models/switch 随时切换<br>
    ✅ 双模型架构：复杂推理走大模型API，日常问答走本地7B
  </p>
</div>
"""))

print("\n💡 配置后验证:")
print("  curl http://192.168.2.xx:18080/api/models/providers")
print("  curl -X POST http://192.168.2.xx:18080/api/models/switch \\")
print('    -H "Content-Type: application/json" \\')
print('    -d \'{"provider": "deepseek", "model": "deepseek-chat"}\'')

## 故障排除

### 常见问题速查

| 问题 | 症状 | 解决 |
|------|------|------|
| **端口占用** | `Address already in use` | `lsof -i :18080` → `kill -9 PID` |
| **Ollama 未运行** | `Connection refused` 访问 11434 | `ollama serve` 启动服务 |
| **模型下载慢** | 进度条卡住不动 | 先用 `--skip-model` 启动框架，再单独 `ollama pull` |
| **Python 版本过低** | 语法错误 / import 失败 | `python3 --version` 确保 >= 3.9 |
| **依赖安装失败** | `pip install` 报错 | 尝试 `pip install --user` 或使用虚拟环境 |
| **设备无法访问** | 浏览器打不开 192.168.2.xx | 检查防火墙/同一网段/服务绑定 0.0.0.0 |
| **内存不足** | OOM / 服务崩溃 | 7B 模型需 8GB+ 内存，考虑关闭其他服务 |

### 调试命令

```bash
# 检查端口
lsof -i :18080 -i :18081 -i :18082 -i :11434

# 查看 Ollama 日志
journalctl -u ollama -f

# 测试 API 连通性
curl -s http://192.168.2.xx:18080/health

# 查看服务日志
python -m framework.api.server --multi-port --debug 2>&1 | tee server.log
```

### 三端口说明

| 端口 | 服务 | 说明 |
|------|------|------|
| 18080 | 终端 HTML | LumiTerminal 交互页面 + 所有 API 端点 |
| 18081 | REST API | 纯 API 服务，无前端，供第三方集成 |
| 18082 | 模型管理 | 模型列表、切换、健康检查专用端口 |

> 💡 **提示**：生产环境建议使用 `start.sh`（Linux）或 `start.bat`（Windows）一键启动，脚本会处理端口检查和后台运行。

## 🏋️ 练习任务

动手验证你的部署，完成以下 5 个任务：

### 任务 1：添加外部模型
添加你的 DeepSeek API 密钥到 `config/providers.yaml`，用 `/api/models/switch` 切换到 DeepSeek 模型，测试对话。

```bash
# 验证切换成功
curl http://192.168.2.xx:18080/api/models
```

### 任务 2：全端点验证
运行 `verify.sh`（或手动 curl）验证所有 API 端点，截图结果。至少验证 7 个端点。

```bash
bash scripts/verify.sh
```

### 任务 3：多设备访问
从手机浏览器访问 `http://192.168.2.xx:18080`，发送一条消息，观察 LumiTerminal 的响应式布局。

### 任务 4：费曼教学 API
用 curl 调用 `/api/feynman/explain`，尝试讲解"概率论"。观察五步教学法（现象→冲突→模型→推导→测试）的输出。

```bash
curl -X POST http://192.168.2.xx:18080/api/feynman/explain \
  -H "Content-Type: application/json" \
  -d '{"topic": "概率论", "subject": "数学"}'
```

### 任务 5：模型参数调优
修改 Ollama Modelfile 的 `temperature` 参数（默认 0.7），重新创建模型，对比不同 temperature 下的回复风格差异。

```bash
# 创建自定义 Modelfile
ollama show qwen2.5:7b --modelfile > Modelfile.custom
# 编辑 temperature 参数
sed -i 's/PARAMETER temperature 0.7/PARAMETER temperature 1.2/' Modelfile.custom
# 创建新模型
ollama create lumilearn-creative -f Modelfile.custom
```

---

| 任务 | 难度 | 技能点 |
|------|------|--------|
| 1. 添加模型 | ⭐⭐ | API 配置、模型路由 |
| 2. 端点验证 | ⭐ | HTTP 请求、API 测试 |
| 3. 多设备 | ⭐ | 网络配置、响应式设计 |
| 4. 费曼API | ⭐⭐ | Prompt 工程、教学引擎 |
| 5. 参数调优 | ⭐⭐⭐ | 模型参数、推理调优 |

> 🎯 **学习建议**：先完成任务 2 确认部署无误，再挑战任务 4 和 5 深入理解 AI 教学引擎和模型参数调优。

---

## 📚 延伸阅读

- [架构文档 (ARCHITECTURE.md)](../ARCHITECTURE.md) — 五层架构完整说明
- [学习路径 (INDEX.md)](../learning_journey/INDEX.md) — 从零到一学习路线
- [开发者指南 (DEVELOPER_GUIDE.md)](../../DEVELOPER_GUIDE.md) — API 开发与贡献